## Whole notebook

In [1]:
from pathlib import Path
from trail_pacer.utils import DATA_PATH

GPX_FILE = Path(DATA_PATH,"gpx", "TSB2026-LA_BOUCLE_CUGEOISE.gpx")

In [2]:
from trail_pacer.utils import gpx_to_dataframe

df = gpx_to_dataframe(GPX_FILE)
print(f"Track points : {len(df)}")
print(f"Total distance : {df['distance_m'].max() / 1000:.2f} km")
print(f"Min elevation : {df['elevation_m'].min():.0f} m")
print(f"Max elevation : {df['elevation_m'].max():.0f} m")
df.head()

Track points : 1218
Total distance : 25.86 km
Min elevation : 185 m
Max elevation : 1035 m


,lat,lon,elevation_m,distance_m
0,43.279237,5.703046,223.24,0.000000
1,43.279237,5.703046,222.66,0.006785
2,43.278894,5.703307,222.26,43.644764
3,43.278930,5.703258,221.61,49.344395
4,43.278837,5.703127,220.66,64.121342


In [3]:
from trail_pacer.utils import interpolate_splits, compute_split_stats

In [ ]:
splits = interpolate_splits(df, step_km=1)
splits

,distance_km,elevation_m
0,0.000000,223.240000
1,1.000000,189.790097
2,2.000000,290.503822
3,3.000000,471.747558
4,4.000000,563.381648
5,5.000000,584.935067
6,6.000000,602.136209
7,7.000000,698.103644
8,8.000000,799.745629
9,9.000000,610.455515


In [5]:
split_stats = compute_split_stats(splits)
split_stats

,distance_km,elevation_m,elev_diff_m,split_gain_m,split_loss_m,cum_gain_m,cum_loss_m,grade_pct
0,0.000000,223.240000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1.000000,189.790097,-33.449903,0.000000,33.449903,0.000000,33.449903,-3.344990
2,2.000000,290.503822,100.713725,100.713725,0.000000,100.713725,33.449903,10.071373
3,3.000000,471.747558,181.243736,181.243736,0.000000,281.957462,33.449903,18.124374
4,4.000000,563.381648,91.634090,91.634090,0.000000,373.591551,33.449903,9.163409
5,5.000000,584.935067,21.553419,21.553419,0.000000,395.144970,33.449903,2.155342
6,6.000000,602.136209,17.201142,17.201142,0.000000,412.346112,33.449903,1.720114
7,7.000000,698.103644,95.967435,95.967435,0.000000,508.313547,33.449903,9.596743
8,8.000000,799.745629,101.641985,101.641985,0.000000,609.955532,33.449903,10.164198
9,9.000000,610.455515,-189.290113,0.000000,189.290113,609.955532,222.740017,-18.929011


### 🏃‍♂️ Riegel Formula for Predicting Race Times

The **Riegel formula** is a simple model used to estimate how long a runner will take to complete a race of a different distance based on a known performance. It assumes that performance decreases at a predictable rate as distance increases.

The formula is:

$
T_2 = T_1 \left(\frac{D_2}{D_1}\right)^{1.06}
$

- $T_1$: known time  
- $D_1$: known distance  
- $T_2$: predicted time  
- $D_2$: target distance  
- **1.06**: fatigue exponent (typical for trained runners)

A higher exponent means the runner slows down more as distance increases. The model works well for most endurance events from 1500 m to the marathon.


In [6]:
from trail_pacer.utils import Conversions as conv

def base_pace_from_flat_times(num_kms,perf: tuple,fatigue_rate: float) -> float:
    RIEGEL_EXP   = 1+fatigue_rate

    goal_km, time_min = perf
    TIME_MIN = conv.chrono_to_min(time_min)
    norm_time = TIME_MIN * (num_kms / goal_km) ** RIEGEL_EXP
    return float(norm_time)

# flat_perf = (5,   "18:35")
# flat_perf = (5,   "23:30")
flat_perf = (10,  "38:51")
# flat_perf = (10,  "49:00")
# flat_perf = (21.1,"1:23:00")
# flat_perf = (21.1,"1:48:00")

num_kms = 26
base_pace_min_per_km = base_pace_from_flat_times(num_kms=num_kms, perf=flat_perf,fatigue_rate=0.06)
print(f"Time for {num_kms} km: "+conv.min_to_chrono(base_pace_min_per_km))

Time for 26 km: 1:46:58


In [7]:
conv.chrono_to_min("38:51")

38.85

In [8]:
conv.chrono_to_min("38:51")/10

3.8850000000000002

In [9]:
conv.min_to_chrono(conv.chrono_to_min("38:51")/10)

'3:53'

In [10]:

import numpy as np
import pandas as pd
from datetime import datetime
class PaceModel:
    """
    Predicts race time and per-split paces for a trail run.

    Parameters
    ----------
    pace_10k_min_per_km : float
        Flat 10 km pace in min/km (e.g. 5.0 for 5:00/km).
    up_sens : float
        Minutes per km added per 1 % of uphill grade.
    down_sens : float
        Minutes per km removed per 1 % of downhill grade.
    fatigue_rate : float
        Minutes per km added per km already covered.
    """

    def __init__(
        self,
        pace_10k_min_per_km: str,
        up_sens: float = 0.05,
        down_sens: float = 0.02,
        fatigue_rate: float = 0.005,
    ):
        self.base_pace    = conv.chrono_to_min(pace_10k_min_per_km)
        self.up_sens      = up_sens
        self.down_sens    = down_sens
        self.fatigue_rate = fatigue_rate

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _adjusted_pace(
        self,
        grade_pct: np.ndarray,
        cum_dist_km: np.ndarray,
        fatigue_rate: float | None = None,
    ) -> np.ndarray:
        """Vectorised pace (min/km) per segment given grade and fatigue."""
        fr = fatigue_rate if fatigue_rate is not None else self.fatigue_rate

        grade_adj = (
            self.up_sens   * np.maximum(grade_pct, 0)   # uphill  → slower
          + self.down_sens * np.minimum(grade_pct, 0)   # downhill → faster (grade is negative)
        )
        fatigue_adj = fr * cum_dist_km

        return np.maximum(self.base_pace + grade_adj + fatigue_adj, 0.5)  # floor at 0.5 min/km

    @staticmethod
    def _split_distances(split_stats: pd.DataFrame) -> np.ndarray:
        """Width of each split segment in km."""
        return split_stats["distance_km"].diff().fillna(0).values

    # ------------------------------------------------------------------
    # Prediction methods
    # ------------------------------------------------------------------

    def predict_flat(self, split_stats: pd.DataFrame) -> float:
        """Total time in minutes — flat pace, no grade or fatigue adjustment."""
        return self.base_pace * split_stats["distance_km"].max()

    def predict_no_fatigue(self, split_stats: pd.DataFrame) -> float:
        """Total time in minutes — grade-adjusted, fatigue ignored."""
        pace = self._adjusted_pace(
            split_stats["grade_pct"].values,
            np.zeros(len(split_stats)),
            fatigue_rate=0.0,
        )
        return (pace * self._split_distances(split_stats)).sum()

    def predict_with_fatigue(self, split_stats: pd.DataFrame) -> float:
        """Total time in minutes — grade-adjusted and fatigue-adjusted."""
        pace = self._adjusted_pace(
            split_stats["grade_pct"].values,
            split_stats["distance_km"].values,
        )
        return (pace * self._split_distances(split_stats)).sum()

    def predict_split_times(
        self,
        split_stats: pd.DataFrame,
        start_time: datetime | None = None,
    ) -> pd.DataFrame:
        """
        Return a DataFrame with predicted pace and times per split.

        Columns added:
          predicted_pace  : adjusted pace for that split (min/km)
          split_time_min  : time to complete that split (min)
          cum_time_min    : cumulative race time at the end of that split
          cum_time_str    : cumulative time formatted as H:MM:SS
          arrival_time    : wall-clock arrival time (only if start_time given)
        """
        df   = split_stats.copy()
        dist = self._split_distances(df)

        df["predicted_pace"] = self._adjusted_pace(
            df["grade_pct"].values,
            df["distance_km"].values,
        )
        df["split_time_min"] = df["predicted_pace"] * dist
        df["cum_time_min"]   = df["split_time_min"].cumsum()
        df["cum_time_str"]   = df["cum_time_min"].apply(self._fmt_time)

        if start_time is not None:
            df["arrival_time"] = start_time + pd.to_timedelta(df["cum_time_min"], unit="m")

        return df

    # ------------------------------------------------------------------
    # Utilities
    # ------------------------------------------------------------------

    @staticmethod
    def _fmt_time(minutes: float) -> str:
        total_s = int(round(minutes * 60))
        h, rem  = divmod(total_s, 3600)
        m, s    = divmod(rem, 60)
        return f"{h}:{m:02d}:{s:02d}"

    def summary(self, split_stats: pd.DataFrame) -> None:
        """Print a quick comparison of the three prediction modes."""
        t_flat    = self.predict_flat(split_stats)
        t_grade   = self.predict_no_fatigue(split_stats)
        t_fatigue = self.predict_with_fatigue(split_stats)

        print(f"Base pace       : {self.base_pace:.2f} min/km")
        print(f"Distance        : {split_stats['distance_km'].max():.1f} km")
        print(f"Flat estimate   : {self._fmt_time(t_flat)}")
        print(f"Grade-adjusted  : {self._fmt_time(t_grade)}")
        print(f"Grade + fatigue : {self._fmt_time(t_fatigue)}")

    def __repr__(self) -> str:
        return (
            f"PaceModel(base_pace={self.base_pace:.2f}, "
            f"up_sens={self.up_sens}, down_sens={self.down_sens}, "
            f"fatigue_rate={self.fatigue_rate})"
        )

In [11]:
from datetime import datetime

model = PaceModel(pace_10k_min_per_km='3:53', up_sens=0.8, down_sens=0.02, fatigue_rate=0.00)

model.summary(split_stats)

splits = model.predict_split_times(split_stats, start_time=datetime(2025, 6, 1, 8, 0))
splits["cum_predicted_pace"] = splits["predicted_pace"].cumsum()
splits[["distance_km", "elevation_m", "grade_pct", "predicted_pace", "cum_time_str", "arrival_time"]]

Base pace       : 3.88 min/km
Distance        : 25.9 km
Flat estimate   : 1:40:24
Grade-adjusted  : 3:36:38
Grade + fatigue : 3:36:38


,distance_km,elevation_m,grade_pct,predicted_pace,cum_time_str,arrival_time
0,0.000000,223.240000,0.000000,3.883333,0:00:00,2025-06-01 08:00:00.000000000
1,1.000000,189.790097,-3.344990,3.816434,0:03:49,2025-06-01 08:03:48.986011614
2,2.000000,290.503822,10.071373,11.940431,0:15:45,2025-06-01 08:15:45.411891900
3,3.000000,471.747558,18.124374,18.382832,0:34:08,2025-06-01 08:34:08.381827002
4,4.000000,563.381648,9.163409,11.214061,0:45:21,2025-06-01 08:45:21.225457554
5,5.000000,584.935067,2.155342,5.607607,0:50:58,2025-06-01 08:50:57.681868704
6,6.000000,602.136209,1.720114,5.259425,0:56:13,2025-06-01 08:56:13.247349606
7,7.000000,698.103644,9.596743,11.560728,1:07:47,2025-06-01 09:07:46.891036374
8,8.000000,799.745629,10.164198,12.014692,1:19:48,2025-06-01 09:19:47.772564144
9,9.000000,610.455515,-18.929011,3.504753,1:23:18,2025-06-01 09:23:18.057750540


# Comparing Predictions with actual times from Pierre's run

In [12]:
import pandas as pd

df_temps_pierre = pd.read_excel("/Users/sauniere/Desktop/Perso/Codes/Running/trail-pacer/data/Temps_Pierre_Cugeoise.xlsx")
for col in ['Time','GAP /km']:
    df_temps_pierre[col] = pd.to_datetime(df_temps_pierre[col], format="%H:%M:%S")
    df_temps_pierre[col] = df_temps_pierre[col].dt.strftime("%H:%M")

df_temps_pierre["pace"] = df_temps_pierre["Time"].apply(conv.chrono_to_min) 
# Add a column with the cumulated pace up to that point
df_temps_pierre["cum_pace"] = df_temps_pierre["pace"].cumsum()
df_temps_pierre["cum_dist"] = df_temps_pierre["Distance km"].cumsum()
df_temps_pierre.head(2)

,Lap,Distance km,Time,GAP /km,Elev,HR,pace,cum_pace,cum_dist
0,1,1.0,05:13,05:16,-22,146,5.216667,5.216667,1.0
1,2,1.0,07:39,04:48,80,157,7.650000,12.866667,2.0


In [15]:
splits.head(2)

,distance_km,elevation_m,elev_diff_m,split_gain_m,split_loss_m,cum_gain_m,cum_loss_m,grade_pct,predicted_pace,split_time_min,cum_time_min,cum_time_str,arrival_time,cum_predicted_pace
0,0.0,223.240000,0.000000,0.0,0.000000,0.0,0.000000,0.00000,3.883333,0.000000,0.000000,0:00:00,2025-06-01 08:00:00.000000000,3.883333
1,1.0,189.790097,-33.449903,0.0,33.449903,0.0,33.449903,-3.34499,3.816434,3.816434,3.816434,0:03:49,2025-06-01 08:03:48.986011614,7.699767


In [23]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["elevation_m"],
        fill="tozeroy", fillcolor="rgba(100,150,255,0.15)",
        line=dict(color="rgba(100,150,255,0.6)", width=1.5),
        name="Elevation (m)",
        hovertemplate="%{y:.0f} m<extra></extra>",
    ),
    secondary_y=False,
)

splits["pace_fmt"]    = splits["predicted_pace"].apply(conv.min_to_chrono)

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=splits["predicted_pace"],
        line=dict(color="tomato", width=2.5),
        name="Predicted Pace",
        customdata=splits["pace_fmt"],
        hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=df_temps_pierre["cum_dist"], y=df_temps_pierre["pace"],
        line=dict(color="darkorange", width=2, dash="dot"),
        name="Real Pace",
        # customdata=df_temps_pierre["fatigue_fmt"],
        # hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Elevation & Predicted Pace Profile",
    xaxis_title="Cumulative Distance (km)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.update_yaxes(title_text="Elevation (m)",    secondary_y=False)
fig.update_yaxes(title_text="Pace (min/km)",    secondary_y=True,  autorange="reversed")

fig.show()
# fig.write_html("data/chart_elevation_pace.html", include_plotlyjs="cdn", full_html=False)
# print("✅ Embeddable div saved → data/chart_elevation_pace.html")

In [22]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"], y=df["elevation_m"],
        fill="tozeroy", fillcolor="rgba(100,150,255,0.15)",
        line=dict(color="rgba(100,150,255,0.6)", width=1.5),
        name="Elevation (m)",
        hovertemplate="%{y:.0f} m<extra></extra>",
    ),
    secondary_y=False,
)

splits["pace_fmt"]    = splits["predicted_pace"].apply(conv.min_to_chrono)

fig.add_trace(
    go.Scatter(
        x=splits["distance_km"]+1, y=splits["predicted_pace"],
        line=dict(color="tomato", width=2.5),
        name="Predicted Pace",
        customdata=splits["pace_fmt"],
        hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=df_temps_pierre["cum_dist"], y=df_temps_pierre["pace"],
        line=dict(color="darkorange", width=2, dash="dot"),
        name="Real Pace",
        # customdata=df_temps_pierre["fatigue_fmt"],
        # hovertemplate="%{customdata} min/km<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Elevation & Predicted Pace Profile",
    xaxis_title="Cumulative Distance (km)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.update_yaxes(title_text="Elevation (m)",    secondary_y=False)
fig.update_yaxes(title_text="Pace (min/km)",    secondary_y=True,  autorange="reversed")

fig.show()
# fig.write_html("data/chart_elevation_pace.html", include_plotlyjs="cdn", full_html=False)
# print("✅ Embeddable div saved → data/chart_elevation_pace.html")

In [ ]:
# DIST_COL = "distance_km"
LEN_COL = "distance_km"
GAIN_COL = "split_gain_m"
LOSS_COL = "split_loss_m"
GRADE_COL = "grade_pct"
NET_COL = "net_gain_m"

In [ ]:
df = split_stats.copy()

In [30]:
def naismith_time_minutes(
    distance_km: pd.Series,
    gain_m: pd.Series,
    flat_speed_kmh: float = 8.0,   # running flat pace
) -> pd.Series:
    """
    Return estimated split time in minutes using Naismith's Rule.
    One extra hour per 600 m of ascent.
    """
    flat_time_min  = (distance_km / flat_speed_kmh) * 60
    climb_time_min = (gain_m / 600.0) * 60
    return flat_time_min + climb_time_min

df["naismith_min"] = naismith_time_minutes(
    df[LEN_COL], df[GAIN_COL], flat_speed_kmh=8.0
)
df["naismith_pace_min_km"] = df["naismith_min"] / df[LEN_COL]

print(f"Predicted finish time (Naismith): "
      f"{df['naismith_min'].sum() / 60:.2f} h  "
      f"({int(df['naismith_min'].sum() // 60)}h "
      f"{int(df['naismith_min'].sum() % 60)}min)")

Predicted finish time (Naismith): 46.34 h  (46h 20min)


In [ ]:
def minetti_cost(grade_decimal: float) -> float:
    """Energy cost in J/(kg·m) for a given grade (Minetti et al. 2002)."""
    i = grade_decimal
    return (155.4 * i**5 - 30.4 * i**4 - 43.3 * i**3
            + 46.3 * i**2 + 19.5 * i + 3.6)

FLAT_COST = minetti_cost(0.0)   # ≈ 3.6 J/(kg·m)

def minetti_pace_multiplier(grade_pct: pd.Series) -> pd.Series:
    """Pace multiplier vs flat running (>1 = slower, <1 = faster)."""
    grade_dec = grade_pct / 100.0
    cost = grade_dec.apply(minetti_cost)
    return (cost / FLAT_COST).clip(0.5, 4.0)

df["minetti_multiplier"] = minetti_pace_multiplier(df[GRADE_COL])

# Visualise the Minetti curve
grades = np.linspace(-0.35, 0.35, 200)
costs  = [minetti_cost(g) for g in grades]
multipliers = [c / FLAT_COST for c in costs]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(grades * 100, costs, color="steelblue", linewidth=2)
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("Grade (%)")
axes[0].set_ylabel("Energy cost J/(kg·m)")
axes[0].set_title("Minetti Metabolic Cost Curve")
axes[0].grid(True, linestyle="--", alpha=0.4)

axes[1].plot(grades * 100, multipliers, color="tomato", linewidth=2)
axes[1].axhline(1.0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("Grade (%)")
axes[1].set_ylabel("Pace multiplier (relative to flat)")
axes[1].set_title("Minetti Pace Multiplier")
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("data/minetti_curve.png", dpi=150)
plt.show()